# Advice API Tool
adviceApiTool.py is an implementation of baseApiTool.py which works for the health.gov API.
This document will explain the reasoning behind the overwritten functions and show how baseApiTool.py is implemented.
To see the parent class, check baseApiTool.ipynb

## Imports

In [ ]:
from typing import Type, Optional
from pydantic import BaseModel, Field
from tools.baseApiTool import BaseAPIToolInput, BaseAPITool
import requests

## Advice API Input
This class extends BaseAPIToolInput and adds the parameters that the health.gov API accepts.
All of the parameters (besides baseUrl) are listed as optional so any number of them can be used by the adviceAgent.

The description of each parameter is carefully written, since they are instructions to the agent on what the input is.
For example, if the `age` parameter's description doesn't say `(Enter as a string)`, then the agent will enter an integer, which breaks the API.

In [ ]:
class AdviceAPIToolInput(BaseAPIToolInput):
    baseUrl: str = Field(
        default="https://odphp.health.gov/myhealthfinder/api/v4/myhealthfinder.json?",
        description="Base URL for the API endpoint."
    )
    
    age: Optional[str] = Field(default=None, description="Age of the person. (Enter as a string)")
    sex: Optional[str] = Field(default=None, description="Sex of the person (male/female).")
    pregnant: Optional[str] = Field(default=None, description="Pregnancy status (yes/no).")
    sexuallyActive: Optional[str] = Field(default=None, description="Sexually active status (yes/no).")
    tobaccoUse: Optional[str] = Field(default=None, description="Tobacco use status (yes/no).")

## AdviceAPITool Class
The description of the API tool is a safeguard to ensure that the agent properly understands the tool.

We overwrite parse response so that we can limit the response given to our LLM.
Here, we give the first section of the first resource back to the agent.

In [ ]:
class AdviceAPITool(BaseAPITool):
    name: str = "APITool"
    description: str = (
        "Use this tool when you need to receive healthcare information. "
        "Provide baseUrl (optional) and any of the following parameters: "
        "age, sex, pregnant, sexuallyActive, tobaccoUse."
    )
    args_schema: Type[BaseModel] = AdviceAPIToolInput

    def parse_response(self, response: requests.Response):
        data = response.json()
        try:
            return data['Result']['Resources']['All']['Resource'][0]['Sections']['section'][0]
        except Exception:
            return data

## Example run
Here is an example run of the health.gov API. This API is free and unlimited.
In this example we are giving healthcare advice to a person that is described by the parameters.
We run the API in an ugly way, but this is done so that it can be run in a nice way for the agentic system.

In [ ]:
apiTool = AdviceAPITool()

query = AdviceAPIToolInput(
    age="35",
    sex="female",
    pregnant="no",
    sexuallyActive="yes",
    tobaccoUse="no"
)

result = apiTool._run(**query.model_dump())

print(result)